# Playground Series S6E8 (Predicting Smartphone Addiction) — `Predicting Smartphone Addiction | Rank-Gauss Stack` 解説付き写し

- **コンペ**: [Predicting Smartphone Addiction (Playground Series - Season 6, Episode 8)](https://www.kaggle.com/competitions/playground-series-s6e8)（3,531チーム / **本日終了**）
- **原著者**: Asterios Terzis
- **元notebook**: https://www.kaggle.com/code/asteriosterzis/predicting-smartphone-addiction-rank-gauss-stack
- **Public Score / Best Score**: **0.97130 / 0.97130**（本日時点でこのコンペの公開最高スコア帯）
- **実行時間**: 17分43秒（CPU）

> ⚠️ これは**学習目的の解説付き写し**です。原著者のコードは変更しておらず、出力は含んでいません。
> 元の英語Markdownセルはそのまま残し、各コードセルの直前に日本語の解説セルを追加しています。

---

## なぜこのnotebookを選んだか

このコンペでは、これまで20本以上のnotebookを本ルーティンで扱ってきましたが、**上位帯のほとんどが「他人の提出CSVをランク平均するだけ」**で、学習価値が薄いものでした。

このnotebookは公開最高スコア（0.97130）を出しながら、**まったく逆のことをしています**。著者は「自分のスコアの改善のほとんどはノイズかもしれない」と自分で疑い、それを**測定で示しています**。

- 241個のモデルをスタックしているが、**そのうち自己参照的なもの（同じプールから作られたブレンド）を明示的に除外**している
- 「public LBで0.00007以下の差はノイズと区別できない」ことを**計算して示す**
- **過去7エピソードの実データ**を使って「public上位10位に入ったチームがprivateでも上位10位に残ったか」を検証している（答え: 3エピソードでは誰も残らなかった）

つまり **「スコアを上げる方法」ではなく「自分のスコアを疑う方法」を教えてくれるnotebook**です。

## 評価指標（metric）

**ROC AUC**（Receiver Operating Characteristic の曲線下面積）。二値分類（スマホ依存かどうか）の指標です。

- 定義: **ランダムに選んだ「陽性の1人」のスコアが、ランダムに選んだ「陰性の1人」のスコアより高くなる確率**。
- 範囲は 0.5（ランダム）〜 1.0（完璧）。
- **順位だけで決まり、絶対値には依存しません。** 予測を単調変換（例: すべて2乗、ランクに変換）してもAUCは変わりません。

**なぜこの指標か**: 依存症の判定はクラス不均衡（陽性が少数）になりがちで、accuracyだと「全員陰性」と答えるだけで高得点が出てしまいます。またしきい値をいくつにすべきかは応用場面によるので、**しきい値に依存しない**指標が適切です。

**このnotebookが指標をどう利用しているか**: AUCが順位のみに依存することを徹底的に活用しています。全メンバーを**パーセンタイル順位**に変換してからスタックし（`pct_rank`）、最後の公開提出とのブレンドも順位空間で行っています。異なるライブラリのモデル（生スコアが数十のオーダーのものもある）を**キャリブレーションの違いを気にせず混ぜられる**のがこの設計の利点です。


# Predicting Smartphone Addiction | Rank-Gauss Stack over 241 Models | LB 0.97130

### Playground S6E8 · ROC AUC · a stack that runs end to end, and the experiments behind each choice

Everything here runs from public inputs. It builds a pool from the published out-of-fold
libraries, fits a logistic meta-model on rank-gauss transformed member predictions, and blends
the result with the strongest public submission. My best submission from this construction
scored **0.97130**.

Three of the things I checked along the way are included as code you can run rather than tables
you have to take my word for: the split-half test that shows why the exact feature value matters,
a simulation of what choosing on the public leaderboard costs you, and what the finished Season 6
private boards actually did to the public top ten.

I got a couple of things wrong first and I have left those in, because the corrections were more
useful than the original guesses.

**Contents**

1. Setup and the fold convention
2. Building the pool
3. Duplicates, and why they are worse than they look
4. Members that are already stacks
5. What to feed the meta-model
6. Fitting the stack
7. Blending with the public submission
8. Submission
9. Why the exact value matters, and everything else I tried
10. What the leaderboard can and cannot resolve
11. What the private board did in the finished episodes

## 1. Setup and the fold convention

Every public OOF library on this competition aligns to the same split:

```python
StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
```

over `train.csv` in original file row order. The `.npy` files carry no ids, so the alignment is
positional. If you sort or reindex the frame before stacking, every number you compute afterwards
is wrong and will still look reasonable.

I only compare things on these same folds. A difference measured on two different splits is not a
difference, it is noise, and at the scale this competition is decided on that matters.

## 【解説 1】セットアップと fold（分割）の規約

**何をしているか (What)**
ライブラリのimportと、このコンペの公開OOFライブラリが共通で使っている分割規約の定義です:

```python
StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
```

**なぜそうするのか (Why)**
これがこのnotebook全体の前提です。他人が公開した `.npy` の予測ファイルには**行IDが入っておらず、位置（行番号）だけで対応**します。つまり全員が `train.csv` の**元のファイル行順**で、**同じseed・同じ分割**を使っていることに全員が暗黙に合意している状態です。

もし1人でも別のseedを使っていたら、その人のOOFは他人と行がずれ、スタックは静かに壊れます。**エラーは出ず、スコアが少し下がるだけ**です。著者がこの規約を冒頭で明示しているのは、この「暗黙の契約」が壊れやすいことを知っているからです。

*用語*:
- **OOF (Out-Of-Fold) 予測** — k分割交差検証で、各foldの「学習に使わなかった部分」に対する予測を集めたもの。学習データ全体分の「未見データに対する予測」が得られるので、スタッキングのメタモデルの入力に使えます。
- **StratifiedKFold** — 各foldでの陽性/陰性の比率が元データと同じになるように分割する方法。不均衡データでは必須。


In [ ]:
import glob
import os
import re

import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

N_TRAIN, N_TEST = 691369, 296302
INPUT = os.environ.get("S6E8_INPUT", "/kaggle/input")


def find_file(filename, must_contain=None):
    '''Locate a file anywhere under the input root. Kaggle mounts competitions and datasets
    under different layouts depending on how they are attached, so search rather than hardcode.'''
    hits = glob.glob(f"{INPUT}/**/{filename}", recursive=True)
    if must_contain:
        hits = [h for h in hits if must_contain.lower() in h.lower()]
    if not hits:
        raise FileNotFoundError(f"{filename!r} (containing {must_contain!r}) not found under {INPUT}")
    return sorted(hits, key=len)[0]


def find_dir(folder):
    '''Locate a dataset directory by name. Returns None if it is not attached.'''
    hits = [d for d in glob.glob(f"{INPUT}/**/{folder}", recursive=True) if os.path.isdir(d)]
    return sorted(hits, key=len)[0] if hits else None


train = pd.read_csv(find_file("train.csv", "playground-series-s6e8"))
test = pd.read_csv(find_file("test.csv", "playground-series-s6e8"))
y = train["addicted_label"].to_numpy()
folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(N_TRAIN), y))

print(f"{len(train):,} train rows, {len(test):,} test rows, positive rate {y.mean():.4f}")

## 2. Building the pool

Several people have published OOF and test prediction pairs on this split. They use a few
different naming conventions (`oof_x.npy` with `test_x.npy`, `testpred_x.npy` or `tep_x.npy`,
plus a couple of parquet tables), so the loader handles all of them.

I check the shape and that everything is finite before accepting a member. A member that is the
wrong length is not a member, it is a silent misalignment.

## 【解説 2】プールの構築 — 公開されたOOF/testペアを読み込む

**何をしているか (What)**
複数の公開データセットに散らばっている「OOF予測 + test予測」のペアを読み込みます。命名規則が人によってバラバラ（`oof_x.npy` に対して `test_x.npy` / `testpred_x.npy` / `tep_x.npy`）なので、ローダーが全パターンを吸収します。読み込み時に**形状（行数）と欠損（NaN）を検査**しています。

**なぜそうするのか (Why)**
`load_pairs` の設計で注目すべきは、**読み込み時点で検証している**ことです。行数が合わないファイルやNaNを含むファイルを黙って受け入れると、後段のメタモデルが壊れるか、あるいは**壊れずに少しだけ悪いスコアを出します**。後者の方が厄介です。

データを集める段階で「使えないものを弾く」のは地味ですが、スタッキングの成否を分けます。


In [ ]:
def load_pairs(root, prefix, test_prefixes=("test_", "testpred_", "tep_")):
    '''Load oof_<name>.npy / test_<name>.npy pairs from a directory tree.'''
    out = {}
    for path in glob.glob(os.path.join(root, "**", "oof_*.npy"), recursive=True):
        name = os.path.basename(path)[4:-4]
        mate = next((c for c in (os.path.join(os.path.dirname(path), tp + name + ".npy")
                                 for tp in test_prefixes) if os.path.exists(c)), None)
        if mate is None:
            continue
        oof, tst = np.load(path).astype(np.float64), np.load(mate).astype(np.float64)
        if (oof.shape == (N_TRAIN,) and tst.shape == (N_TEST,)
                and np.isfinite(oof).all() and np.isfinite(tst).all()):
            out[prefix + name] = (oof, tst)
    return out


SOURCES = [
    ("s6e8-oof-library-47-models", "sz_"),
    ("s6e8-oof-library-11-members", "nn_"),
    ("s6e8-mask-augmented-oof-library", "ma_"),
    ("s6e8-full-best-blend-npy", "tam_"),
    ("s6e8-adarsh-oof-library", "a_"),
    ("s6e8-golem-oof-library", "golem_"),
    ("s6e8-fm-lattice-blend-members", "fm_"),
    ("s6e8-150-fusion-local-members", "hb_"),
    ("s6e8-catstrall-member", "x_"),
    ("s6e8-catstr-aug16", "mk_"),
]

members = {}
for folder, prefix in SOURCES:
    root = find_dir(folder)
    got = load_pairs(root, prefix) if root else {}
    members.update(got)
    print(f"{folder:36s} {len(got):3d}" + ("" if root else "   NOT ATTACHED"))

## 【解説 3】parquet形式のライブラリの取り込み

**何をしているか (What)**
別の公開者（boltuzamaki氏）は1モデル1ファイルではなく、**1つのparquetの1列 = 1モデル**という形式で公開しているので、そちらも読み込んでプールに合流させます。

**なぜそうするのか (Why)**
実務でも同じことが起きます。同じ意味のデータが**複数のフォーマットで**存在するのは普通です。ここでは形式の違いを吸収する層を1か所に閉じ込め、下流からは統一されたプールに見えるようにしています。

*用語*: **Parquet** — 列指向のバイナリデータ形式。同じ列の値が連続して並ぶので圧縮が効き、「特定の列だけ読む」が高速。CSVより桁違いに軽いので、こういう大量の数値列の配布に向いています。


In [ ]:
# boltuzamaki publishes one parquet column per model
bolt = find_dir("s6e8-oof-prediction-library") or ""
if bolt and os.path.exists(f"{bolt}/oof_predictions.parquet"):
    oof_df = pd.read_parquet(f"{bolt}/oof_predictions.parquet")
    tst_df = pd.read_parquet(f"{bolt}/test_predictions.parquet")
    for col in oof_df.columns:
        if col != "id" and col in tst_df:
            members[f"bolt_{col}"] = (oof_df[col].to_numpy(float), tst_df[col].to_numpy(float))

# szymonkapiski's 50 deliberately weak models, published as one (n, 50) array
weak = find_dir("s6e8-50-weakest-oof-models") or ""
if weak and os.path.exists(f"{weak}/oof.npy"):
    WO = np.load(f"{weak}/oof.npy", mmap_mode="r")
    WT = np.load(f"{weak}/test.npy", mmap_mode="r")
    for j in range(WO.shape[1]):
        members[f"weak_{j:02d}"] = (np.asarray(WO[:, j], float), np.asarray(WT[:, j], float))

names = sorted(members)
OOF = np.column_stack([members[n][0] for n in names])
TST = np.column_stack([members[n][1] for n in names])
auc = pd.Series([roc_auc_score(y, OOF[:, j]) for j in range(OOF.shape[1])], index=names)
print(f"{len(names)} members, OOF AUC from {auc.min():.5f} to {auc.max():.5f}")

## 3. Duplicates, and why they are worse than they look

Some arrays are published in more than one dataset under different names. If you glob a directory
and stack whatever you find, those members get counted twice.

That is not a cosmetic problem. A duplicated member gets double weight in the meta-model without
anything in the output telling you so, and the meta-model has no way to know that the two columns
are the same measurement rather than two agreeing measurements.

## 【解説 4】重複メンバーの検出 — 「見た目より悪い問題」

**何をしているか (What)**
全メンバーを**パーセンタイル順位**に変換し（`pct_rank`）、順位相関行列を計算して、ほぼ同一のメンバー（別名で複数のデータセットに公開されている同じ配列）を検出します。

```python
def pct_rank(v):
    return (rankdata(v) - 0.5) / len(v)
```

**なぜそうするのか (Why)**
著者のセクション見出しが「Duplicates, and why they are worse than they look（重複、そしてなぜそれが見た目より悪いのか）」です。

**重複したメンバーは、メタモデルの中で自動的に2倍の重みを持ちます。** 単に無駄なのではなく、**意図しない重み付けを静かに導入する**のが問題です。あなたが「241個のモデルを等しい機会で競わせている」と思っていても、実際には偶然2回公開されたモデルが2票持っている。これはスコアを下げるだけでなく、**「どのモデルが効いているか」の解釈も壊します**。

`pct_rank` の `-0.5` は、順位を `(0,1)` の開区間に収めるための補正です（0や1になると後の正規分位変換で ±∞ になる）。

*用語*: **順位相関 (rank correlation)** — 値そのものではなく順位の一致度を測る相関。AUCが順位のみに依存する以上、重複判定も順位空間で行うのが筋です。


In [ ]:
def pct_rank(v):
    return (rankdata(v) - 0.5) / len(v)


R = np.column_stack([pct_rank(OOF[:, j]) for j in range(OOF.shape[1])]).astype(np.float32)
Rt = np.column_stack([pct_rank(TST[:, j]) for j in range(TST.shape[1])]).astype(np.float32)

# rank correlation matrix as a standardised inner product, so it fits in memory
Z = (R - R.mean(0)) / (R.std(0) + 1e-12)
corr = (Z.T @ Z) / len(Z)
del Z

drop = set()
for i in range(len(names)):
    if names[i] in drop:
        continue
    for j in range(i + 1, len(names)):
        if names[j] in drop or corr[i, j] <= 0.9995:
            continue
        drop.add(names[j] if auc[names[i]] >= auc[names[j]] else names[i])

print(f"{len(drop)} near-duplicate members dropped at rank correlation > 0.9995")

## 4. Members that are already stacks

This is the thing I nearly got wrong, and it is the part of this notebook I would most want
someone else to check.

Some published members are not models. They are blends fitted over this same pool. Their OOF is
therefore optimistic in-sample, and a meta-model that sees them will over-weight them. The
strongest single member sits at OOF 0.9702; the strongest member that is actually a model sits at
0.9689. That gap of 0.0013 is not a better model, it is the difference between a prediction and a
prediction of a prediction.

I noticed because a greedy hill-climb over 231 members picked three of them and put 85% of the
weight on one. I wrote that down as a finding — something like "228 of these members are
redundant" — and it was wrong. When I removed all 26 self-referential members and refit:

| | members | OOF | LB |
|---|---|---|---|
| with them | 252 | 0.970255 | 0.97124 |
| without them | 239 | 0.970152 | 0.97122 |

Cross-validation dropped by 0.0001 and the leaderboard moved by 0.00002. So they were
contributing inflated CV rather than predictive power, and the hill-climb result was an artifact
of that inflation rather than a fact about the pool. The pre-stacked member was a convenient
packaging of the pool, not a substitute for it.

They are excluded below. The CV number you get is lower and I trust it more.

## 【解説 5】自己参照的メンバーの除去 — このnotebookで最も重要な処理

**何をしているか (What)**
正規表現でメンバー名を検査し、`naji`, `sz_naji`, `v13_anchor`, `hb_candidate` で始まるものをプールから除外します。

```python
SELF_REFERENTIAL = re.compile(r"^(naji|sz_naji|v13_anchor|hb_candidate)")
```

**なぜそうするのか (Why)**
著者自身が「これは私が危うく間違えたところで、他の人に一番検証してほしい部分だ」と書いています。

問題はこうです。公開されているメンバーの一部は**モデルではなく、この同じプールに対して既にフィットされたブレンド**です。そのブレンドのOOF予測は、**同じ行を使って重みを決めた結果**なので、**in-sampleで楽観的**です。

それをメタモデルの入力に入れると何が起きるか。メタモデルは「このメンバーは異常に当たる」と学習し、大きな重みを与えます。しかしその当たりは実力ではなく**自分自身の答えを覗き見していた**だけなので、**testでは再現しません**。OOFスコアは上がり、LBは上がらない。典型的なリークです。

これは**スタッキングにおける最も見落とされやすい失敗モード**です。「公開されているOOFだから安全」ではありません。**そのOOFがどうやって作られたかを問わなければならない。**

*用語*: **リーク (leakage)** — 本来使えないはずの情報が学習に混入し、検証スコアだけが不当に良くなること。


In [ ]:
SELF_REFERENTIAL = re.compile(r"^(naji|sz_naji|v13_anchor|hb_candidate)")

keep = [i for i, n in enumerate(names) if n not in drop and not SELF_REFERENTIAL.match(n)]
names = [names[i] for i in keep]
R, Rt = R[:, keep], Rt[:, keep]
print(f"{len(names)} members kept")

## 5. What to feed the meta-model

Members arrive on very different calibrations — one library ships raw factorization-machine
scores in the tens — so the usual choices are percentile ranks, logits, or both.

I measured all of them on two folds, paired. Pairing matters: fold 0 of this split is about
0.0007 harder than fold 3, which is ten times any effect I was looking for, so an unpaired
comparison would have been meaningless. That sweep ran on a 252-member version of the pool, so
the column counts are from that run rather than the one this notebook builds:

| design | columns | mean AUC |
|---|---|---|
| logits only | 252 | 0.969942 |
| ranks only | 252 | 0.969972 |
| **rank-gauss only** | **252** | **0.970024** |
| ranks + logits | 504 | 0.970008 |
| rank-gauss + logits | 504 | 0.970006 |

The ordering `logits < ranks < rank-gauss` held in both folds separately, which is what convinced
me it was a real effect and not one lucky fold.

Ranks discard calibration, which is what you want here, but they are uniform, so they compress
the tails — and the tails are where AUC is decided. Pushing the ranks through the normal quantile
gives that resolution back without reintroducing any dependence on how each member happened to be
calibrated. Adding logits on top of rank-gauss makes it slightly worse. So the final design is
rank-gauss alone: half the columns, and about seven times faster to fit.

## 6. Fitting the stack

Logistic regression, cross-fitted, so no weight is ever fitted on a row it scores.

Two details that are not cosmetic. `StandardScaler` is required — without it lbfgs does not
converge at any sane `max_iter`, and a non-converged fit reads *higher* than the truth, which is
the worst possible failure mode. So the code asserts convergence rather than hoping for it.

Regularisation barely matters here: `C` from 0.01 to 3.5 moves the score by less than 0.00003 on
this design. With 553,000 training rows and 241 columns the meta-model is not variance-limited.
Where it does matter is on wider designs — a missingness-interaction design with about 930 columns
needed `C = 0.01` and gained 0.00008 from it.

## 【解説 6】Rank-Gauss 変換とスタックのフィッティング

**何をしているか (What)**
2段階です。

1. **Rank-Gauss変換**: パーセンタイル順位 `R` を、正規分布の分位点関数 `norm.ppf` に通します。これで各メンバーの分布が**標準正規分布に揃います**。
2. **クロスフィッティングされたロジスティック回帰**: 各foldについて、そのfoldを除いたデータでメタモデルを学習し、そのfoldを予測します。`StandardScaler` を必ず通し、`max_iter=3000`, `tol=1e-5` を指定。

**なぜそうするのか (Why)**

- **Rank-Gauss を使う理由**: メンバーごとにキャリブレーションが全然違います（あるライブラリは生の factorization machine スコアで数十のオーダー）。順位に直せばスケールは揃いますが、順位は一様分布なので線形モデルには扱いにくい。**正規分位変換をかけると、外れ値の影響を消したまま、線形モデルが好む形になります**。これが "Rank-Gauss" と呼ばれる前処理です。
- **`StandardScaler` が必須な理由（著者が強調）**: これがないと lbfgs ソルバは現実的な `max_iter` で収束しません。そして著者が指摘する恐ろしい点 —— **収束していないフィットは、真の値より「高い」スコアを出す**。つまり `ConvergenceWarning` を見逃すと、実力より良く見える結果が手に入り、それを信じてしまう。警告を無視してはいけない典型例です。
- **クロスフィットの理由**: メタモデルの重みが、それが採点する行から決して学習されないようにするため。これを怠るとOOFスコアが楽観的になります。

*用語*: **Rank-Gauss (RankGauss)** — 順位→正規分位への変換。ニューラルネットの表形式データ前処理として Michael Jahrer が広めた手法。`sklearn` の `QuantileTransformer(output_distribution="normal")` とほぼ同じ。


In [ ]:
G = norm.ppf(np.clip(R, 1e-7, 1 - 1e-7)).astype(np.float32)
Gt = norm.ppf(np.clip(Rt, 1e-7, 1 - 1e-7)).astype(np.float32)


def fit_logistic(X_fit, y_fit, X_pred, C=1.0):
    scaler = StandardScaler().fit(X_fit)
    model = LogisticRegression(C=C, max_iter=3000, solver="lbfgs", tol=1e-5)
    model.fit(scaler.transform(X_fit), y_fit)
    assert int(np.max(model.n_iter_)) < 3000, "meta-model did not converge"
    return model.predict_proba(scaler.transform(X_pred))[:, 1]


oof_meta = np.zeros(N_TRAIN)
for fit_idx, val_idx in folds:
    oof_meta[val_idx] = fit_logistic(G[fit_idx], y[fit_idx], G[val_idx])

print(f"stack OOF AUC = {roc_auc_score(y, oof_meta):.6f}")
test_meta = fit_logistic(G, y, Gt)

## 7. Blending with the public submission

The stack on its own scored 0.97125 on the leaderboard, from a 248-member version of the pool.
Blending it with the strongest public submission does better, so the question is how much of each.

I mapped the whole curve instead of guessing a weight, because the shape tells you whether what
you are looking at is real:

| weight on my stack | LB |
|---|---|
| 0.00 | 0.97128 |
| 0.10 | 0.97129 |
| 0.15 | 0.97129 |
| **0.20** | **0.97130** |
| **0.30** | **0.97130** |
| **0.40** | **0.97130** |
| **0.50** | **0.97130** |
| 0.65 | 0.97129 |
| 1.00 | 0.97125 |

Four consecutive weights give the same score and it falls away symmetrically on both sides. A
broad plateau is worth much more than a single high point: if the maximum had been a spike at one
weight I would have assumed it was noise and ignored it. I take the middle of the plateau rather
than an edge of it, and `0.35` is not tuned, it is just the middle.

A few days earlier I ran the same curve with a weaker version of this stack and it was
monotonically *decreasing* — every bit of my own model made the public score worse. What changed
in between was adding the mask-augmented members, which also improved cross-validation on all five
folds. Two independent measurements moving the same way is the only reason I believe this one.

## 【解説 7】公開提出とのブレンド — 重みを「推測」せず「地図化」する

**何をしているか (What)**
自分のスタックと、最強の公開提出を **W = 0.35** で順位空間ブレンドします。

```python
final = pct_rank(W * pct_rank(test_meta) + (1 - W) * public)
```

**なぜそうするのか (Why)**
著者は「重みを推測する代わりに曲線全体を地図化した」と書いています。つまり W を 0〜1 で振ってスコアの形を見た上で 0.35 を選んでいます。

ここで注目すべきは **W = 0.35 という値そのものではなく、著者がこの直後（解説9・10）でこの選択自体を疑っていること**です。「曲線のピークを取る」という行為は、**public LBという有限のサンプルに対する最適化**であり、それ自体が過学習になりうる。普通のnotebookはピークを取って終わりますが、このnotebookはそこから検証を始めます。


In [ ]:
s1 = pd.read_csv(find_file("submission.csv", "vault"))["addicted_label"].to_numpy()
s2 = pd.read_csv(find_file("submission (1).csv", "vault"))["addicted_label"].to_numpy()
public = pct_rank((2.9 * s1 + 0.1 * s2) / 3.0)

W = 0.35
final = pct_rank(W * pct_rank(test_meta) + (1 - W) * public)
print(f"rank correlation, stack vs public submission: {np.corrcoef(pct_rank(test_meta), public)[0, 1]:.5f}")

## 8. Submission

## 【解説 8】提出ファイルの生成とアサーション

**何をしているか (What)**
`submission.csv` を書き出す前に4つの `assert` を置いています: 行数が正しいか、IDが一意か、値が有限（NaN/infでない）か、値が[0,1]に収まっているか。

**なぜそうするのか (Why)**
これも label-free な自己検査です。**間違った提出ファイルは、低いスコアではなくエラーとして返ってきます**（あるいは最悪、無効化されます）。4行のassertで防げるなら書くべきです。

`assert` は「起こるはずのないこと」を明示するドキュメントでもあります。読む人に「ここでは行数とIDの一意性が保証されている」と伝わります。


In [ ]:
submission = pd.DataFrame({"id": test["id"], "addicted_label": final})
assert len(submission) == N_TEST
assert submission["id"].is_unique
assert np.isfinite(submission["addicted_label"]).all()
assert submission["addicted_label"].between(0, 1).all()
submission.to_csv("submission.csv", index=False)
submission.head()

## 9. Why the exact value matters, and everything else I tried

Early on I put monotone constraints on the screen-time features. More screen time really does mean
more addiction on average, so constraining the model that way should have been free. It cost
0.0017, which is large by the standards of anything else in this competition.

That failure is a diagnosis. The signal here is not a smooth function of magnitude. This data is
synthetic, generated from a small original dataset, and the generator memorised value-to-label
associations. The exact value acts as a lookup key.

The cell below is the check. `notifications_per_day` has univariate AUC of about 0.51 — no
monotone signal at all. But if you split the training data in half at random and compute, for each
exact value, how far its positive rate sits from the base rate, the two halves agree almost
perfectly. The permuted-label control is there so you can see that the method does not manufacture
the correlation by itself.

## 【解説 9】単調制約の失敗と「値ごとの効果」の分析

**何をしているか (What)**
学習データを乱数で半分に分け、各カラムの**値ごとの陽性率が全体の基準率からどれだけ離れているか**を計算します（`value_effect`）。最小カウント30、スムージング20で希少値を安定化しています。

**なぜそうするのか (Why)**
著者はこう書いています。「最初、スクリーンタイム系の特徴量に**単調制約**を入れた。スクリーンタイムが長いほど依存度が高いのは事実だから、この制約はタダで効くはずだった。ところが 0.0017 も**下がった**。」

理由が次のMarkdownセルで明かされます。上位3カラムは**単調な信号を持っておらず**、代わりに「値ごとの効果」が**片方の半分ともう片方の半分で r > 0.94 で再現する**——つまりこれは合成データ特有の、**値そのものに紐づいた非単調なパターン**です。単調制約はこれを平滑化して捨ててしまった。

**教訓**: ドメイン知識に基づく制約（「常識的にこうなるはず」）が、データの実際の構造と矛盾することがあります。**制約を入れたら必ず測る。** 「理屈が通っているから正しいはず」で済ませない。

*用語*:
- **単調制約 (monotone constraint)** — 「この特徴量が増えたら予測は必ず増える（減らない）」と勾配ブースティングに強制する設定。LightGBM/XGBoostの `monotone_constraints`。過学習を抑え解釈性を上げる目的で使われます。
- **スムージング (smoothing)** — 出現回数の少ないカテゴリの推定値を全体平均に寄せる処理。ターゲットエンコーディングでの必須テクニック。


In [ ]:
rng = np.random.default_rng(0)
half = rng.random(N_TRAIN) < 0.5
PRIOR, MIN_COUNT, SMOOTH = y.mean(), 30, 20


def value_effect(col, mask, target):
    '''For each exact value: how far its positive rate sits from the base rate.'''
    d = pd.DataFrame({"v": train[col][mask], "y": target[mask]}).dropna()
    g = d.groupby("v")["y"].agg(["mean", "size"])
    g = g[g["size"] >= MIN_COUNT]
    return (g["mean"] * g["size"] + PRIOR * SMOOTH) / (g["size"] + SMOOTH) - PRIOR


def split_half_r(col, target):
    a, b = value_effect(col, half, target), value_effect(col, ~half, target)
    common = a.index.intersection(b.index)
    return np.corrcoef(a[common], b[common])[0, 1], len(common)


y_shuffled = rng.permutation(y)
rows = []
for col in ["age", "notifications_per_day", "app_opens_per_day", "sleep_hours",
            "gaming_hours", "work_study_hours", "social_media_hours",
            "weekend_screen_time", "daily_screen_time_hours"]:
    present = train[col].notna()
    a = roc_auc_score(y[present], train[col][present])
    r, n_values = split_half_r(col, y)
    r0, _ = split_half_r(col, y_shuffled)
    rows.append({"column": col, "distinct values": n_values,
                 "univariate AUC": round(max(a, 1 - a), 3),
                 "split-half r": round(r, 3), "shuffled control": round(r0, 3)})

pd.DataFrame(rows)

The three columns at the top have no usable monotone signal and per-value effects that reproduce
at r above 0.94. Anything that imposes smoothness throws that away, which is what the monotone
constraints did.

Everything else, measured on the same folds:

| idea | result |
|---|---|
| target-encode every column at its exact value | **+0.0034** |
| add pair encodings for all 36 numeric pairs | included in the above |
| monotone constraints on screen-time features | −0.0017 |
| logistic regression on the raw features | −0.033 |
| impute the missing values before the GBDT | +0.0006 |
| bigger trees (255 or 511 leaves) | −0.0006 |
| round the lattice to one decimal instead of exact values | −0.0031 |
| my own seven GBDTs added to the pool | +0.000016, 4 folds of 5 |
| 21 further public members added to the pool | +0.000005, mixed signs |
| the nine mask-augmented members added | **+0.000028, 5 folds of 5** |
| per-missingness-regime rank calibration | −0.0017 |
| missingness-regime interactions in the meta | roughly neutral |
| greedy hill-climb instead of logistic regression | −0.0001 |
| equal-weight average of five subset stacks | −0.00007 |
| nine public submissions equally weighted instead of three | −0.00017 on LB |

Two of these are worth pulling out.

**Equal weighting is not the neutral choice.** I assumed averaging more public submissions with
equal weights would be safe, since I was not fitting anything. It cost 0.00017, which is above the
noise floor. Equal weights are a choice, and they over-reward the weak members.

**"Diversity beats strength" does not mean what I first read it to mean.** It is a well-known
result on this pool and I applied it to the wrong question. Selecting the 50 most decorrelated
members scores 0.969655; selecting the 50 strongest scores 0.969820. The principle is about the
marginal value of *adding* a member to an already saturated pool, not about which subset to pick
when building one. And no subset beat simply using all of them, which is what the code above does.

## 10. What the leaderboard can and cannot resolve

The public split is about 20% of the test set, roughly 59,000 labels. Two submissions correlated
at 0.99 differ by about 0.00007 from noise alone. Most of the differences being chased at the top
of this board, mine included, are smaller than that.

Rather than argue about it, here is the experiment. The situation to simulate is the one that
actually arises near the top of a board: choosing between two candidates that are nearly
equivalent. So I fit the same stack twice with different regularisation, which gives two honest
predictions correlated above 0.999 — the kind of pair whose ordering the leaderboard is asked to
decide every day.

Hold out 59,260 rows at random as a pretend public leaderboard. Choose the blend weight that
maximises AUC there. Then score that choice on the 632,109 rows you never touched, and compare it
with the weight you would have picked using all the data.

This is a deliberately gentle search: one signal, 21 candidate weights, no post-processing. A real
leaderboard search over notebooks, seeds, bands and blend partners carries a far larger
multiple-testing burden.

## 【解説 10】public LBの分解能を測る — このnotebookの白眉

**何をしているか (What)**
シミュレーションです。

1. 同じ設計で正則化だけ大きく変えた（`C=0.01`）**もう1つのスタック**を作る。この2つは「ほぼ同等」だと分かっている。
2. `PUBLIC_ROWS = 59,260` 行（public LBの実際のサイズ、test全体の約20%）をランダムに選ぶ試行を **40回**繰り返す。
3. 各試行で「そのpublic行だけを見て最良のブレンド重みを選ぶ」ことをして、**選んだ重みが、残りの行（つまり本当のprivate相当）でどれだけ良かったか**を測る。

**なぜそうするのか (Why)**
著者の結論（次のMarkdownセル）:

> **見せかけのpublicボード上で得た改善の半分以上は、使わなかった行に触れた瞬間に消える。そしてそうやって選んだ重みは、大多数の試行において「正直な」重みより悪い。**

これは**public LBの分解能を実測している**ということです。相関0.99の2つの提出は、**ノイズだけで約0.00007の差が出ます**。このコンペの上位陣が争っている差（0.97125 vs 0.97130 = 0.00005）は、**その分解能より小さい**。

著者は同時に、この主張の限界も明示しています。「2つの候補が本当に違うなら、public boardはそれを検出できる。言っているのは**publicボードを使って重みを選ぶ**行為が危ない、ということだ。」——過剰な一般化を自分で戒めているのが誠実です。

**一般化できる教訓**: 検証セットのサイズから、**自分が検出できる最小の差**を計算しておくべきです。それより小さい差を追いかけているなら、あなたはノイズを最適化しています。

*用語*: **分解能 (resolution)** — 測定系が区別できる最小の差。統計的には標準誤差に相当します。


In [ ]:
PUBLIC_ROWS, N_TRIALS = 59260, 40
GRID = np.arange(0.0, 1.01, 0.05)

# a second, near-equivalent stack: same design, much stronger regularisation
oof_alt = np.zeros(N_TRAIN)
for fit_idx, val_idx in folds:
    oof_alt[val_idx] = fit_logistic(G[fit_idx], y[fit_idx], G[val_idx], C=0.01)

a, b = pct_rank(oof_meta), pct_rank(oof_alt)
print(f"candidate A OOF = {roc_auc_score(y, a):.6f}")
print(f"candidate B OOF = {roc_auc_score(y, b):.6f}")
print(f"rank correlation = {np.corrcoef(a, b)[0, 1]:.5f}\n")

honest_w = max(GRID, key=lambda w: roc_auc_score(y, w * a + (1 - w) * b))
sim = []
for _ in range(N_TRIALS):
    idx = rng.permutation(N_TRAIN)
    pub, priv = idx[:PUBLIC_ROWS], idx[PUBLIC_ROWS:]
    scores = [roc_auc_score(y[pub], w * a[pub] + (1 - w) * b[pub]) for w in GRID]
    w_sel = GRID[int(np.argmax(scores))]
    sim.append({
        "gain seen on pretend public": max(scores) - roc_auc_score(y[pub], b[pub]),
        "gain realised on the rest": (roc_auc_score(y[priv], w_sel * a[priv] + (1 - w_sel) * b[priv])
                                      - roc_auc_score(y[priv], b[priv])),
        "vs choosing honestly": (roc_auc_score(y[priv], w_sel * a[priv] + (1 - w_sel) * b[priv])
                                 - roc_auc_score(y[priv], honest_w * a[priv] + (1 - honest_w) * b[priv])),
    })

sim = pd.DataFrame(sim)
print(f"honest weight over all {N_TRAIN:,} rows: {honest_w:.2f}\n")
print(f"mean gain visible on the pretend public board : {sim.iloc[:, 0].mean():+.6f}")
print(f"mean gain actually realised on the rest       : {sim.iloc[:, 1].mean():+.6f}")
print(f"public-chosen weight worse than the honest one: "
      f"{(sim.iloc[:, 2] < 0).sum()} of {N_TRIALS} trials")

More than half of the gain you see on the pretend public board does not survive contact with the
rows you did not use, and the weight chosen that way is worse than the honest one in the large
majority of trials. Note what this is and is not saying: when two candidates genuinely differ, the public
board does find it. The problem is that near the top of this competition they do not genuinely
differ, and the board is being asked to resolve a difference smaller than its own noise.

This is not hypothetical for me. I improved my cross-validation by a small amount on all five
folds and watched the public leaderboard go *down* by 0.00008 on the same change. If I had been
selecting on the leaderboard I would have thrown away the better model.

## 11. What the private board did in the finished episodes

Seven Season 6 episodes have completed, so their public and private boards are an unusually
relevant backtest. The question is narrow: if a team was in the public top ten, did it stay in the
private top ten?

## 【解説 11】終了済み7エピソードでのバックテスト

**何をしているか (What)**
Season 6 で既に終了した7エピソードの public/private リーダーボードの実データを読み、エピソードごとに「public上位10位のチームが private でどこにいたか」を集計します。

**なぜそうするのか (Why)**
これはシミュレーションではなく**実際に起きたこと**です。結果（最後のMarkdownセル）:

> **7エピソード中4つで、public 1位のチームは private で379位より下だった。3つのエピソードでは、public 上位10位から誰1人 private 上位10位に残らなかった。**

これが最も強い証拠です。前セルのシミュレーションは「理屈の上ではこうなるはず」でしたが、こちらは**Playground Seriesという同じ舞台で、実際に何度も起きていること**を示しています。

著者は caveat も付けています。「エピソードごとに指標が違うので、スプレッドの比較はエピソード内でのみ有効」。**自分の証拠の限界を自分で述べる**のは、良い分析の条件です。

**このnotebookから持ち帰るべきこと**: 高スコアnotebookを読むとき、「どうやってスコアを上げたか」と同じくらい **「著者がそのスコアをどれだけ疑っているか」**を見るべきです。疑っていない高スコアは、大抵publicへの過学習です。


In [ ]:
lb_dir = find_dir("playground-series-s6-leaderboards")
if lb_dir:
    lb = pd.read_csv(f"{lb_dir}/s6_leaderboards.csv")
    eps = pd.read_csv(f"{lb_dir}/s6_episodes.csv").set_index("episode")["metric"]
    rows = []
    for ep in sorted(lb.episode.unique()):
        d = lb[(lb.episode == ep) & (~lb.is_host_baseline)].dropna(
            subset=["public_rank", "private_rank"])
        top10, top100 = d[d.public_rank <= 10], d[d.public_rank <= 100]
        winner = d[d.public_rank == 1]
        rows.append({
            "episode": ep,
            "metric": eps.get(ep, "?"),
            "teams": len(d),
            "public top 10 kept": f"{(top10.private_rank <= 10).sum()}/10",
            "private rank of public #1": int(winner.private_rank.iloc[0]) if len(winner) else None,
            "top 100 spread": round(top100.public_score.max() - top100.public_score.min(), 5),
            "median rank move": int(top100.rank_delta.abs().median()),
        })
    display(pd.DataFrame(rows))
else:
    print("attach georgymamarin/playground-series-s6-leaderboards to run this")

In four of seven episodes the public winner finished below rank 379, and in three of them nobody
from the public top ten survived into the private top ten.

The last two columns are the interesting part, with one caveat: the episodes do not all share a
metric, so spreads are only comparable within a metric. Restricting to the three ROC AUC episodes
makes the pattern clean. S6E2 had its whole top 100 inside 0.00009 and the median team moved 409
places; S6E3 and S6E5 had four to five times that spread and moved 39 and 135. The size of the
shakeup tracks how tightly packed the public scores were, not how hard people tried.
I also checked whether making more submissions predicted losing rank, and it does not — the
correlation is about −0.07, which is nothing.

So this is not a story about overfitters being punished. It is simpler than that: when the top of
the board is tied to within its own noise, the private ordering is close to a lottery, and no
amount of probing changes that.

The current S6E8 top 100 spans about 0.0005, which puts it closer to the episodes that held than
the ones that scrambled. That is mildly reassuring and I would not lean on it.

## What I would do differently

Measure the noise floor first. I spent several days on changes of 0.00001 before working out that
the leaderboard cannot see them, and the only way I found anything real after that was to stop
looking at the leaderboard and start looking at per-fold signs.

Prefer a wide plateau to a high point, everywhere, not just in the blend weight.

And when a result looks like a finding, try to break it before writing it down. Both of the things
I got wrong here — the hill-climb picking three members, and the subset selection — looked like
clean results until I ran the control.

Credit to @szymonkapiski, @boltuzamaki, @dariushafshar, @raykkretzschmar, @adarsh1077, @najiama,
@paiky1995, @aadijoshi19, @tamerlanomralinov, @hboyang, @anthonytherrien and @georgymamarin, whose
published arrays and leaderboard data this is built on. @aadijoshi19's mask-augmented members were
the only addition in a week that improved my cross-validation on all five folds.